# Spambase : détection de spam

**Mini-projet Machine Learning — TAF MCE (IMT Atlantique)**

Ce notebook applique le workflow défini dans `ml_workflow.py` au jeu **Spambase** (UCI) :
1. import des données ; 2. nettoyage et prétraitement ; 3. séparation train / test ;
4. entraînement avec sélection de variables ; 5. validation.

Toutes les fonctions appelées ici sont définies dans `ml_workflow.py` (préfixe `mw.`).

## 0. Le problème et le choix de la métrique

On veut décider si un e-mail est un spam (classe 1) à partir de 57 statistiques sur son texte.
Les deux erreurs n'ont pas le même coût. La documentation du jeu le dit explicitement :
*« False positives (marking good mail as spam) are very undesirable »*. Un faux positif fait
disparaître un vrai e-mail dans le dossier spam ; un faux négatif laisse passer une publicité.

C'est exactement le cadre du **test de Neyman-Pearson** vu en cours : on fixe un budget d'erreur
de type I (faux positif) $\alpha$, et on maximise la puissance (le rappel) sous cette contrainte.
On retient donc :

| Rôle | Métrique |
|---|---|
| Comparer les modèles indépendamment du seuil | **AUC-ROC** |
| Critère métier (Neyman-Pearson) | **rappel à taux de faux positifs ≤ 1 %** (`tpr_fpr1`) |
| Lisibilité (le jeu est presque équilibré) | accuracy, précision, F1 |

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_workflow as mw

SEED = 42
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 90

## 1. Import et contrôle des données

In [ ]:
df_raw = mw.load_spambase("data")
print(df_raw.shape)
mw.describe_dataset(df_raw, "is_spam")

- **Pas de valeur manquante** : l'étape d'imputation reste dans le pipeline (elle sera testée
  à la section 4.2 en retirant artificiellement des valeurs).
- Le sujet annonce « 5000 entrées, 50 features » : les vrais chiffres sont **4 601 lignes et 57 variables**.
- **391 lignes sont des doublons exacts.** Ici, une ligne dupliquée est très probablement le
  même e-mail (57 fréquences continues identiques au centième près). Si on les garde, un même
  e-mail peut tomber à la fois dans le train et dans le test, et le score de test devient optimiste.
  **On les supprime avant la séparation** : c'est la seule opération autorisée avant le split,
  car elle n'apprend rien des données.
- Après suppression, 3 profils restent présents avec les deux labels : ils sont gardés (bruit d'étiquetage irréductible).

In [ ]:
df = mw.remove_duplicates(df_raw)
X, y = mw.split_features_target(df, "is_spam")
print(f"Après suppression des doublons : {X.shape[0]} e-mails, {X.shape[1]} variables, {y.mean():.1%} de spams")
mw.plot_class_balance(y, labels=("non-spam", "spam"));

Le jeu est **à peu près équilibré** (40 % de spams). Le classifieur trivial « toujours non-spam »
obtient 60 % d'accuracy : c'est la référence à battre.

### Nature des variables

In [ ]:
desc = pd.DataFrame({
    "part_de_zeros": (X == 0).mean(),
    "moyenne": X.mean(),
    "max": X.max(),
    "asymetrie": X.skew(),
})
print("Part moyenne de zéros :", f"{(X == 0).values.mean():.1%}")
desc.sort_values("max", ascending=False).head(8)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5))
for j, col in enumerate(["word_freq_free", "char_freq_!", "capital_run_length_average"]):
    for c, lab in [(0, "non-spam"), (1, "spam")]:
        v = X.loc[(y == c) & (X[col] > 0), col]          # valeurs non nulles seulement
        axes[0, j].hist(v, bins=50, alpha=0.6, label=lab)
        axes[1, j].hist(np.log1p(v), bins=50, alpha=0.6, label=lab)
    axes[0, j].set_title(f"{col} (valeurs > 0)")
    axes[1, j].set_title(f"log(1 + {col})")
axes[0, 0].legend()
plt.tight_layout()

Trois constats qui guident le prétraitement :

1. **Les échelles sont sans commune mesure** : les fréquences sont des pourcentages (≤ 100),
   alors que `capital_run_length_total` monte à 15 841. Sans centrage-réduction, toute distance
   (kNN, SVM) ou toute pénalité (régression logistique régularisée) serait dominée par cette variable.
2. **Les variables sont très creuses** (environ 3/4 de zéros) et **très asymétriques** (queues lourdes).
   La transformation $x \mapsto \log(1+x)$ écrase ces queues (ligne du bas). On s'attend à ce qu'elle
   aide les modèles linéaires et à distance, et qu'elle soit **sans effet sur les arbres**, qui ne
   dépendent que de l'ordre des valeurs (une transformation croissante ne change aucun seuil).
3. L'**hypothèse gaussienne** des modèles génératifs du cours (LDA, QDA) est clairement fausse sur
   ces données brutes (masse en 0 + queue lourde). On mesurera ce que cela coûte.

## 2. Séparation train / test

La séparation est faite **avant** tout apprentissage (centrage, ACP, sélection), et elle est
**stratifiée** pour garder 40 % de spams dans les deux parties. Le test (20 %) n'est utilisé
qu'**une seule fois**, à la toute fin (section 7). Tous les choix intermédiaires (prétraitement,
modèle, hyperparamètres, variables, seuil) sont faits par **validation croisée 5 plis stratifiée** sur le train.

In [ ]:
X_train, X_test, y_train, y_test = mw.split_data(X, y, test_size=0.2, seed=SEED)
cv = mw.make_cv(5, SEED)
print(f"train : {X_train.shape}, {y_train.mean():.3f} de spams | test : {X_test.shape}, {y_test.mean():.3f} de spams")

SCORING = {
    "auc_roc": "roc_auc",
    "tpr_fpr1": mw.tpr_at_fpr_scorer(0.01),
    "accuracy": "accuracy",
    "precision": "precision",
    "f1": "f1",
}

## 3. Prétraitement

### 3.1 Centrage-réduction codé à la main

`mw.StandardScalerManual` calcule $\mu$ et $\sigma$ **sur le train seulement** et les applique au
test. On vérifie qu'il donne exactement le même résultat que `sklearn.preprocessing.StandardScaler`
(c'est aussi l'objet d'un test unitaire).

In [ ]:
from sklearn.preprocessing import StandardScaler
ours = mw.StandardScalerManual().fit(X_train)
print("Identique à sklearn :", np.allclose(ours.transform(X_test), StandardScaler().fit(X_train).transform(X_test)))
Z_test = ours.transform(X_test)
print(f"Sur le TEST : moyennes dans [{Z_test.mean(0).min():.2f}, {Z_test.mean(0).max():.2f}], "
      f"écarts-types dans [{Z_test.std(0).min():.2f}, {Z_test.std(0).max():.2f}]")

Sur le test, les moyennes ne sont pas exactement nulles ni les écarts-types exactement 1 : c'est
normal et c'est le signe qu'on n'a **pas** utilisé le test pour apprendre $\mu$ et $\sigma$.
Les écarts-types très grands sur certaines variables viennent des queues lourdes (une variable
quasi nulle sur le train peut prendre de grandes valeurs sur quelques e-mails du test).

### 3.2 Valeurs manquantes : moyenne ou médiane ?

Le jeu est complet, mais la consigne demande de gérer les valeurs manquantes. On retire au hasard
10 % puis 30 % des valeurs (au train et au test), on impute par la moyenne ou par la médiane
**apprise sur le train**, et on mesure l'AUC-ROC sur le test pour deux modèles.
(Cette expérience regarde le test, mais elle ne sert à aucun choix du modèle final : elle sert
seulement à illustrer l'imputation.)

In [ ]:
for name, model in [("Régression logistique", mw.LogisticRegressionGD()),
                    ("Gradient Boosting", mw.get_models(SEED)["Gradient Boosting (hist.)"])]:
    res = mw.missing_values_experiment(
        lambda s: mw.build_pipeline(model, impute=s, log_transform=True),
        X_train, y_train, X_test, y_test, rates=(0.0, 0.1, 0.3), seed=SEED)
    print(name); display(res)

- Sans valeur manquante, les deux stratégies sont identiques (rien à imputer).
- Avec 30 % de valeurs manquantes, la **médiane est meilleure** pour les deux modèles (régression
  logistique : 0,971 contre 0,963 d'AUC ; boosting : 0,978 contre 0,975). À 10 %, elle est meilleure
  pour la régression logistique et équivalente pour le boosting.
- **Pourquoi** : pour la plupart des variables, 3/4 des valeurs sont nulles et la distribution a une
  queue lourde. La médiane vaut donc 0 (la valeur la plus probable pour un e-mail), alors que la moyenne
  est tirée vers le haut par quelques valeurs extrêmes : imputer par la moyenne revient à inventer des
  occurrences de mots qui n'existent pas dans un e-mail typique.
- Même avec 30 % de trous, la perte reste d'environ 0,01 d'AUC : les variables sont redondantes
  (plusieurs mots « commerciaux » portent la même information).

**Choix retenu : imputation par la médiane** (inactive ici puisque le jeu est complet).

## 4. ACP (codée à la main)

`mw.PCAManual` diagonalise la matrice de covariance $C = \frac{1}{n-1}X_c^\top X_c = V\Lambda V^\top$
des données centrées-réduites (après $\log(1+x)$) et projette sur les $k$ premiers vecteurs propres.
Les valeurs propres $\lambda_i$ sont les variances le long des axes.

In [ ]:
from sklearn.decomposition import PCA
prep = mw.build_pipeline(None, log_transform=True)      # imputation -> log1p -> centrage-réduction
Z_train = prep.fit_transform(X_train)
pca = mw.PCAManual(n_components=0.95).fit(Z_train)
ref = PCA(n_components=pca.n_components_).fit(Z_train)
print("Variances expliquées identiques à sklearn :", np.allclose(pca.explained_variance_, ref.explained_variance_))
print("Axes identiques au signe près :", np.allclose(np.abs(pca.components_), np.abs(ref.components_)))
print("Signes des 10 premiers axes, nous vs sklearn :",
      np.sign((pca.components_[:10] * ref.components_[:10]).sum(1)).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mw.plot_pca_variance(pca, 0.95, ax=axes[0])
mw.plot_pca_2d(pca.transform(Z_train), y_train, labels=("non-spam", "spam"), ax=axes[1])
plt.tight_layout()

- Notre ACP donne exactement les mêmes variances et les mêmes axes que scikit-learn. Les signes
  coïncident parce qu'on a adopté la même convention (la plus grande composante de chaque axe est
  rendue positive) ; sans convention, un axe et son opposé sont des solutions également valables
  (si $v$ est vecteur propre, $-v$ aussi).
- **Il faut 47 axes sur 57 pour garder 95 % de la variance**, et le premier n'en explique que 13 %.
  La variance est très étalée : les variables sont peu redondantes **linéairement**, donc l'ACP
  comprime mal ce jeu.
- La projection sur PC1–PC2 montre une séparation partielle (les spams occupent une « branche »),
  avec un fort recouvrement. C'est attendu : l'ACP ne regarde pas le label ; les directions de grande
  variance ne sont pas forcément celles qui séparent les classes. L'effet de l'ACP sur la
  classification est mesuré à la section 6.4.

## 5. Comparaison des modèles

On compare des modèles de **familles différentes**, tous dans un pipeline
imputation → (log) → centrage-réduction → modèle, réentraîné dans chaque pli :

| Famille | Modèle | Vu en cours |
|---|---|---|
| Référence | classe majoritaire | |
| Linéaire discriminatif | **régression logistique** (descente de gradient codée à la main) | oui |
| Génératif gaussien | **LDA** et **QDA** (codés à la main), Bayes naïf | oui (TP classification bayésienne) |
| Non paramétrique | kNN | oui |
| Noyau | SVM RBF | |
| Arbres | arbre de décision, Random Forest, Gradient Boosting | |
| Réseau de neurones | MLP à une couche cachée | oui (TP réseaux de neurones) |

Avant de les utiliser, on vérifie que nos implémentations donnent les mêmes résultats que scikit-learn.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
Ztr, Zte = Z_train, prep.transform(X_test)
gd = mw.LogisticRegressionGD(C=1.0).fit(Ztr, y_train)
sk = LogisticRegression(C=1.0, max_iter=10000, tol=1e-10).fit(Ztr, y_train)
print(f"Régression logistique : {gd.n_iter_} itérations, écart max des coefficients avec sklearn = "
      f"{np.abs(gd.coef_ - sk.coef_).max():.1e}, prédictions identiques sur le test : "
      f"{(gd.predict(Zte) == sk.predict(Zte)).mean():.1%}")
lda = mw.GaussianBayesClassifier("shared", reg=0).fit(Ztr, y_train)
lda_sk = LinearDiscriminantAnalysis(solver="lsqr").fit(Ztr, y_train)
print(f"LDA : prédictions identiques à sklearn sur le test : {(lda.predict(Zte) == lda_sk.predict(Zte)).mean():.1%}")

plt.figure(figsize=(5, 3))
plt.plot(np.arange(len(gd.loss_history_)) * 10, gd.loss_history_)
plt.xlabel("itération"); plt.ylabel("coût J(w, b)"); plt.yscale("log")
plt.title("Descente de gradient (Nesterov) : décroissance du coût");

### 5.1 Effet de la transformation log(1 + x)

In [ ]:
models = mw.get_models(SEED)
res_raw = mw.cross_validate_models({k: mw.build_pipeline(m) for k, m in models.items()},
                                   X_train, y_train, SCORING, cv, verbose=False)
res_log = mw.cross_validate_models({k: mw.build_pipeline(m, log_transform=True) for k, m in models.items()},
                                   X_train, y_train, SCORING, cv, verbose=False)
comparison = pd.DataFrame({
    "AUC brut": res_raw["auc_roc_moy"], "AUC log": res_log["auc_roc_moy"],
    "rappel@TFP1% brut": res_raw["tpr_fpr1_moy"], "rappel@TFP1% log": res_log["tpr_fpr1_moy"],
}).sort_values("AUC log", ascending=False)
comparison

- La transformation $\log(1+x)$ **améliore tous les modèles qui ne sont pas des arbres** : AUC de la
  régression logistique 0,966 → 0,977 et surtout rappel à 1 % de faux positifs 0,49 → 0,67 ;
  LDA 0,951 → 0,972 ; kNN 0,961 → 0,974 ; SVM 0,969 → 0,976. En écrasant les queues lourdes, elle évite
  que quelques valeurs extrêmes dominent les distances et les produits scalaires, et elle rend les
  distributions moins éloignées de l'hypothèse gaussienne de la LDA.
- Elle est **sans effet sur les arbres** (boosting : 0,9849 dans les deux cas), comme prévu : une
  transformation strictement croissante ne change pas l'ordre des valeurs, donc pas les partitions
  possibles. (Le léger écart de la forêt vient seulement des seuils placés au milieu de deux valeurs,
  qui ne tombent pas au même endroit après transformation.)

**Choix retenu** : `log1p` pour tous les modèles non arborescents. Résultats détaillés avec `log1p` :

In [ ]:
cols = ["auc_roc_moy", "auc_roc_std", "tpr_fpr1_moy", "tpr_fpr1_std", "accuracy_moy", "precision_moy",
        "f1_moy", "auc_roc_train", "temps_fit_s"]
res_log[cols]

Lecture du tableau (moyenne ± écart-type sur 5 plis) :

- **Tous les modèles battent nettement la référence** (accuracy 60 %, AUC 0,5).
- **Les ensembles d'arbres dominent** : Gradient Boosting (AUC 0,985) et Random Forest (0,982), avec
  environ 80 % de spams détectés à 1 % de faux positifs. Ils captent des interactions et des seuils
  (« beaucoup de `!` **et** de majuscules ») qu'un modèle linéaire ne représente pas.
- Viennent ensuite, très proches, MLP, SVM RBF, régression logistique, kNN et LDA (AUC 0,972 à 0,980).
  La régression logistique (0,977) et la LDA (0,972) ont toutes les deux une frontière linéaire ;
  la LDA est un peu moins bonne parce qu'elle **suppose** des classes gaussiennes de même covariance,
  alors que la régression logistique modélise directement $P(Y=1\mid x)$ sans hypothèse sur $X$.
- **QDA et Bayes naïf sont les plus faibles** : l'hypothèse gaussienne est fausse (masse en 0,
  queue lourde). La QDA estime en plus une covariance de $57\times58/2 = 1\,653$ paramètres par classe
  avec environ 1 340 spams : sans le terme de régularisation `reg`, la matrice ne serait même pas inversible.
  Le Bayes naïf a un rappel à 1 % **nul** alors que son AUC est de 0,94 : ses probabilités, produits de
  57 vraisemblances supposées indépendantes, saturent à exactement 0 ou 1 pour beaucoup d'e-mails : en
  validation, 85 % des spams **et 5,5 % des non-spams** reçoivent le score 1,0 exactement. Aucun seuil ne peut
  alors séparer ces e-mails, et le budget de 1 % de faux positifs est dépassé dès le premier seuil. Illustration
  concrète de ce que coûte une hypothèse d'indépendance fausse : le classement reste correct, les probabilités ne le sont pas.
- **Un arbre seul** (AUC 0,941) est nettement battu par la forêt (0,982) : moyenner des arbres
  décorrélés (bootstrap + tirage des variables) réduit la variance.
- **Sur-apprentissage** : kNN (pondéré par la distance) et les ensembles ont une AUC d'entraînement de 1,000 ou 0,999,
  mais ce sont aussi les modèles qui généralisent le mieux ; c'est l'écart en validation qui compte.
- **Le critère à 1 % de faux positifs est bruité** (écart-type de 0,06 à 0,10) : un pli de validation
  contient environ 500 non-spams, donc 1 % ≈ 5 e-mails. On regarde donc aussi l'AUC, plus stable, et on
  ne sur-interprète pas les écarts de quelques points.
- Tous les entraînements durent moins d'une seconde : le temps n'est pas un critère sur ce jeu.

### 5.2 Biais et variance : courbe de validation du kNN et courbe d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
knn_pipe = mw.build_pipeline(mw.clone(models["kNN"]).set_params(weights="uniform"), log_transform=True)
mw.plot_validation_curve(knn_pipe, X_train, y_train, "model__n_neighbors", [1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200],
                         "roc_auc", cv, log_x=True, ax=axes[0])
axes[0].set_title("kNN : AUC-ROC en fonction de k")
mw.plot_learning_curve(mw.build_pipeline(models["Gradient Boosting (hist.)"]), X_train, y_train, "roc_auc", cv,
                       ax=axes[1], title="Gradient Boosting : courbe d'apprentissage")
plt.tight_layout()

- **kNN** : avec $k=1$, AUC de 1,000 en entraînement et 0,91 en validation (variance maximale : chaque point
  est son propre voisin). La validation progresse jusqu'à un plateau vers $k \approx 10$–$50$ (0,97), puis
  redescend lentement : quand $k$ devient grand, la décision moyenne des zones trop larges (biais).
  Notre choix par défaut $k=15$ est sur le plateau.
- **Gradient Boosting** : l'AUC de validation augmente encore avec la taille du train (0,964 avec 270
  e-mails, 0,985 avec 2 700) et l'écart avec l'entraînement se réduit. Le modèle est plutôt **limité par la variance** :
  plus d'e-mails l'amélioreraient encore un peu.

### 5.3 Réglage des hyperparamètres des meilleurs candidats

Recherche par grille, **critère = rappel à 1 % de faux positifs** (le critère métier), en validation croisée 5 plis.

In [ ]:
grids = {
    "Gradient Boosting (hist.)": (mw.build_pipeline(models["Gradient Boosting (hist.)"]),
        {"model__learning_rate": [0.05, 0.1], "model__max_leaf_nodes": [15, 31, 63],
         "model__l2_regularization": [0.0, 1.0], "model__max_iter": [500]}),
    "Random Forest": (mw.build_pipeline(models["Random Forest"]),
        {"model__max_features": ["sqrt", 0.2, 0.33], "model__min_samples_leaf": [1, 2, 5],
         "model__n_estimators": [300]}),
    "SVM RBF": (mw.build_pipeline(models["SVM RBF"], log_transform=True),
        {"model__C": [0.3, 1, 3, 10, 30], "model__gamma": ["scale", 0.003, 0.01, 0.03]}),
    "Régression logistique": (mw.build_pipeline(models["Régression logistique"], log_transform=True),
        {"model__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}),
    "MLP (1 couche cachée)": (mw.build_pipeline(models["MLP (1 couche cachée)"], log_transform=True),
        {"model__hidden_layer_sizes": [(32,), (64,), (128,), (64, 32)], "model__alpha": [1e-4, 1e-3, 1e-2, 1e-1]}),
}
tuned, tuned_rows = {}, []
for name, (pipe, grid) in grids.items():
    best, table = mw.tune_model(pipe, grid, X_train, y_train, mw.tpr_at_fpr_scorer(0.01), cv)
    tuned[name] = best
    tuned_rows.append({"modele": name, "meilleurs_parametres": table.iloc[[0], :-2].to_dict("records")[0],
                       "rappel@TFP1%_CV": table.loc[0, "mean_test_score"], "ecart_type": table.loc[0, "std_test_score"]})
pd.set_option("display.max_colwidth", None)
tuning_summary = pd.DataFrame(tuned_rows).set_index("modele").sort_values("rappel@TFP1%_CV", ascending=False)
tuning_summary

- Le réglage améliore surtout le SVM (rappel à 1 % : 0,71 → 0,75), le boosting (0,79 → 0,81) et la forêt
  (0,80 → 0,83) ; la régression logistique gagne peu (0,67 → 0,68) car $C$ ne change que le degré de lissage
  d'une frontière qui reste linéaire.
- **Random Forest et Gradient Boosting sont statistiquement indistinguables** : 0,826 ± 0,052 contre
  0,809 ± 0,069, soit un écart bien inférieur à un écart-type. On applique la règle fixée à l'avance
  (meilleur score moyen en validation croisée) : **Random Forest**. Le boosting serait un choix tout aussi défendable.

## 6. Sélection de variables

### 6.1 Pourquoi la sélection doit être faite *dans* la validation croisée

Expérience classique : 100 individus, 5 000 variables de **bruit pur**, labels tirés au hasard.
L'accuracy honnête est donc 50 %. (a) On garde les 50 variables les plus corrélées au label sur
**tout** le jeu, puis on fait une validation croisée ; (b) la sélection est une étape du pipeline,
refaite dans chaque pli.

In [ ]:
mw.selection_leakage_demo(n_samples=100, n_features=5000, k=50, seed=0)

En (a), la sélection a « vu » les plis de validation : elle a choisi des variables qui, par hasard,
sont corrélées au label **sur ces individus-là**, et la CV annonce plus de 90 % d'accuracy sur du
bruit. C'est pourquoi toutes nos sélections sont des étapes de `Pipeline` (filtre, L1, RFE).

### 6.2 Classement des variables par quatre critères (filtre, filtre, embarqué, permutation)

In [ ]:
X_train_log = np.log1p(X_train)          # transformation monotone : ne change que ANOVA et L1
ranking = mw.rank_features(X_train_log, y_train, seed=SEED)
mw.plot_feature_ranking(ranking, top=20)
print("Variables mises à zéro par la pénalité L1 (C = 0,05) :", int((ranking["abs_coef_L1"] == 0).sum()), "sur", len(ranking))
ranking[["anova_F", "info_mutuelle", "abs_coef_L1", "permutation_RF", "rang_moyen"]].head(12)

- **Accord des critères sur le haut du classement** : `char_freq_!`, `char_freq_$`, `remove`, `free`, `000`,
  `money`, `hp` et les longueurs de suites de majuscules. C'est le vocabulaire attendu d'un spam commercial.
- **Les désaccords sont instructifs** :
  - `capital_run_length_total` est 8e pour l'ANOVA mais **38e par permutation** : elle est corrélée à
    `_longest` et `_average`. Quand on la mélange, la forêt s'appuie sur les deux autres et le score
    ne baisse pas. L'importance par permutation **sous-estime les variables redondantes**.
  - `your` est 2e pour l'ANOVA mais 17e pour la L1 et la permutation : elle est très liée à la classe
    prise seule, mais redondante avec les autres mots commerciaux une fois ceux-ci dans le modèle.
    Un filtre évalue chaque variable isolément ; la L1 et la permutation évaluent sa contribution **en plus des autres**.
- **Biais du jeu** : `hp` et `george` sont parmi les variables les plus utiles pour la L1 (rangs 1 et 2).
  Elles signalent des **non-spams** parce que le jeu vient de la boîte mail d'un employé de HP Labs prénommé
  George. Un modèle qui s'appuie dessus apprend cette boîte mail, pas le spam en général : il ne
  généraliserait pas tel quel à une autre messagerie.
- La pénalité L1 met 18 coefficients sur 57 **exactement à zéro** : c'est la sélection « embarquée ».

### 6.3 Score en fonction du nombre de variables gardées

Filtre par **information mutuelle** (`SelectKBest`) placé dans le pipeline, pour la régression logistique et le Gradient Boosting.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from functools import partial
mi = partial(mutual_info_classif, random_state=SEED)
ks = [2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 57]
best_lr = tuned["Régression logistique"].named_steps["model"]
best_gb = tuned["Gradient Boosting (hist.)"].named_steps["model"]
curve_lr = mw.cv_score_vs_k(lambda k: mw.build_pipeline(best_lr, log_transform=True, selector=SelectKBest(mi, k=k)),
                            X_train, y_train, ks, "roc_auc", cv)
curve_gb = mw.cv_score_vs_k(lambda k: mw.build_pipeline(best_gb, selector=SelectKBest(mi, k=k)),
                            X_train, y_train, ks, "roc_auc", cv)
ax = mw.plot_score_vs_k(curve_lr, "AUC-ROC", label="régression logistique")
mw.plot_score_vs_k(curve_gb, "AUC-ROC", ax=ax, label="gradient boosting")
ax.legend();

L'AUC monte très vite jusqu'à environ 15–20 variables, puis continue de croître doucement jusqu'à 57.
Retirer des variables **n'améliore pas** le score ici : avec 3 368 e-mails pour 57 variables, le
sur-apprentissage dû à la dimension est déjà contenu par la régularisation (C pour la régression
logistique, sous-échantillonnage et profondeur pour le boosting). La sélection reste utile pour
la parcimonie : avec 20 variables sur 57, on perd environ 0,005 d'AUC.

### 6.4 Comparaison : toutes les variables, sélection (filtre, embarquée L1, wrapper RFE) ou ACP

In [ ]:
from sklearn.feature_selection import SelectFromModel, RFECV
from sklearn.linear_model import LogisticRegression

l1_selector = SelectFromModel(LogisticRegression(penalty="l1", C=0.05, solver="liblinear", max_iter=5000,
                                                   random_state=SEED))
rfe_selector = RFECV(LogisticRegression(max_iter=5000), step=2, cv=mw.make_cv(3, SEED), scoring="roc_auc", n_jobs=-1)
variants = {
    "LR — 57 variables": mw.build_pipeline(best_lr, log_transform=True),
    "LR — filtre MI (k=25)": mw.build_pipeline(best_lr, log_transform=True, selector=SelectKBest(mi, k=25)),
    "LR — embarquée L1": mw.build_pipeline(best_lr, log_transform=True, selector=l1_selector),
    "LR — wrapper RFECV": mw.build_pipeline(best_lr, log_transform=True, selector=rfe_selector),
    "LR — ACP 95 %": mw.build_pipeline(best_lr, log_transform=True, pca=0.95),
    "GB — 57 variables": mw.build_pipeline(best_gb),
    "GB — filtre MI (k=25)": mw.build_pipeline(best_gb, selector=SelectKBest(mi, k=25)),
    "GB — ACP 95 %": mw.build_pipeline(best_gb, log_transform=True, pca=0.95),
    "kNN — 57 variables": mw.build_pipeline(models["kNN"], log_transform=True),
    "kNN — filtre MI (k=25)": mw.build_pipeline(models["kNN"], log_transform=True, selector=SelectKBest(mi, k=25)),
    "kNN — ACP 95 %": mw.build_pipeline(models["kNN"], log_transform=True, pca=0.95),
}
res_sel = mw.cross_validate_models(variants, X_train, y_train, SCORING, cv, verbose=False)
res_sel[["auc_roc_moy", "auc_roc_std", "tpr_fpr1_moy", "accuracy_moy", "temps_fit_s"]].sort_index()

In [ ]:
# Nombre de variables retenues par la sélection L1 et par RFECV (sur tout le train)
lr_l1 = variants["LR — embarquée L1"].fit(X_train, y_train)
lr_rfe = variants["LR — wrapper RFECV"].fit(X_train, y_train)
kept_l1 = X_train.columns[lr_l1.named_steps["select"].get_support()]
kept_rfe = X_train.columns[lr_rfe.named_steps["select"].get_support()]
print(f"L1 garde {len(kept_l1)} variables, RFECV en garde {len(kept_rfe)}.")
print("Écartées par L1 :", sorted(set(X_train.columns) - set(kept_l1)))

- **Aucune réduction de dimension ne bat le modèle complet** de manière significative : les écarts
  sont inférieurs à un écart-type.
- **Régression logistique** : la sélection embarquée L1 (39 variables) et le wrapper RFECV (43 variables)
  font aussi bien que les 57 variables (AUC ≈ 0,977) avec un tiers de variables en moins : ce sont les
  meilleures versions parcimonieuses. Le filtre par information mutuelle (k = 25) perd un peu plus,
  car il note chaque variable seule et garde des variables redondantes.
- **ACP à 95 %** : 47 composantes, donc presque pas de réduction, et pas de gain. Pour le boosting elle
  **dégrade** le critère métier (rappel à 1 % : 0,81 → 0,77) : les arbres coupent selon les axes d'origine
  (« fréquence de `!` > s »), et l'ACP mélange les variables en combinaisons sans sens pour un seuil.
- Les variables écartées par la L1 sont surtout des mots rares ou liés au contexte du donateur
  (`415`, `650`, `857`, `telnet`, `labs` : indicatifs et vocabulaire de HP).

**Choix retenu** : on garde les 57 variables pour le modèle final (meilleur score, et les ensembles
d'arbres gèrent bien les variables peu utiles). La sélection sert à l'interprétation et fournit une
alternative linéaire parcimonieuse (régression logistique + L1).

## 7. Choix du modèle, du seuil, puis évaluation finale sur le test

### 7.1 Choix du seuil de décision par le critère de Neyman-Pearson

Le seuil 0,5 correspond à la règle MPE (erreur totale minimale) si les probabilités sont bien
calibrées ; il ne tient pas compte de l'asymétrie des coûts. On choisit plutôt le seuil
**de Neyman-Pearson** : le plus petit seuil tel que le taux de faux positifs reste ≤ 1 %.
Pour ne pas toucher au test, ce seuil est calculé sur les **scores hors pli** du train
(chaque e-mail du train est scoré par un modèle qui ne l'a pas vu).

In [ ]:
FINAL_NAME = tuning_summary.index[0]      # meilleur critère métier en validation croisée (section 5.3)
final_model = tuned[FINAL_NAME]
oof = mw.out_of_fold_scores(final_model, X_train, y_train, cv)
thr_np = mw.neyman_pearson_threshold(y_train, oof, alpha=0.01)
print(f"Modèle retenu : {FINAL_NAME}")
print(f"Seuil de Neyman-Pearson (TFP ≤ 1 % hors pli sur le train) : {thr_np:.3f}")
mw.plot_threshold_tradeoff(y_train, oof, {"seuil 0,5 (MPE)": 0.5, "seuil NP (α = 1 %)": thr_np});

### 7.2 Évaluation finale (le test n'est utilisé qu'ici)

In [ ]:
final_model.fit(X_train, y_train)
test_default = mw.evaluate_on_test(final_model, X_test, y_test)
test_np = mw.evaluate_on_test(final_model, X_test, y_test, threshold=thr_np)
cols = ["auc_roc", "accuracy", "precision", "rappel", "taux_faux_positifs", "f1", "FP", "FN"]
display(mw.results_table({f"{FINAL_NAME}, seuil 0,5": test_default, f"{FINAL_NAME}, seuil NP": test_np}, cols))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
mw.plot_confusion(test_default, ("non-spam", "spam"), ax=axes[0], title="seuil 0,5")
mw.plot_confusion(test_np, ("non-spam", "spam"), ax=axes[1], title=f"seuil NP = {thr_np:.2f}")
plt.tight_layout()

- **Seuil 0,5** : 94,7 % d'accuracy, mais 17 vrais e-mails sur 506 envoyés dans les spams (3,4 % de faux
  positifs), bien au-dessus du budget de 1 %.
- **Seuil de Neyman-Pearson (0,76)** : **2 faux positifs sur 506 (0,4 %)**, donc la contrainte de 1 % est
  respectée sur des données jamais vues, avec **81 % des spams détectés** et une précision de 99,3 %.
  Le prix est de laisser passer 36 spams de plus : c'est exactement le compromis voulu.
- L'AUC de test (0,989) et le rappel au seuil NP (0,81) sont cohérents avec la validation croisée
  (0,98 et 0,83) : les choix faits sur le train n'ont pas sur-appris.
- Le taux de faux positifs réel (0,4 %) est en dessous de la cible : le seuil a été estimé sur environ
  2 000 non-spams du train, soit seulement une vingtaine d'e-mails au-delà du seuil. Cette estimation est
  imprécise, et un seuil un peu prudent n'est pas surprenant.

Pour situer le modèle retenu, on affiche aussi les courbes ROC et PR sur le test des autres candidats
réglés. **Ce graphique n'a servi à aucun choix** : le modèle avait été choisi en validation croisée.

In [ ]:
test_scores = {}
for name, model in tuned.items():
    model.fit(X_train, y_train)
    test_scores[name] = mw.get_scores(model, X_test)
lda_pipe = mw.build_pipeline(mw.GaussianBayesClassifier("shared", reg=1e-3), log_transform=True).fit(X_train, y_train)
test_scores["LDA (à la main)"] = mw.get_scores(lda_pipe, X_test)
mw.plot_roc_pr(test_scores, y_test, mark_fpr=0.01, roc_xmax=0.2);

## 8. Conclusion

**Modèle retenu : Random Forest (300 arbres) sur les 57 variables, avec un seuil de Neyman-Pearson
($\alpha$ = 1 %) choisi sur les scores hors pli du train.** Sur le test : AUC 0,989, 0,4 % de faux
positifs, 81 % des spams détectés.

Pourquoi ce choix :
1. **Performance** : avec le Gradient Boosting, c'est le meilleur sur le critère métier en validation croisée.
   L'écart entre les deux n'est pas significatif ; le boosting serait un choix équivalent.
2. **Le seuil compte autant que le modèle** : au seuil par défaut, le même modèle fait 8 fois plus de faux positifs.
   Le critère de Neyman-Pearson traduit directement l'exigence de la documentation (« false positives are very undesirable »).
3. **Coût** : moins d'une seconde d'entraînement ; ce n'est pas un critère discriminant ici.

Si l'**interprétabilité** prime, la **régression logistique sur $\log(1+x)$ avec sélection L1** (39 variables)
est l'alternative : AUC de 0,977 en validation croisée, coefficients lisibles en termes de rapports de cotes,
mais environ 15 points de rappel en moins à 1 % de faux positifs.

Ce qu'on retient du workflow sur ce jeu :
- prétraitement : la **forme des distributions** (queues lourdes) compte plus que l'échelle ; `log1p` est
  le prétraitement le plus utile pour les modèles non arborescents, et inutile pour les arbres ;
- l'**ACP** n'aide pas (47 axes sur 57 pour 95 % de la variance) ;
- la **sélection de variables** n'améliore pas le score mais permet un modèle linéaire 30 % plus petit, à performance égale ;
- les modèles **génératifs gaussiens** (QDA, Bayes naïf) souffrent de l'hypothèse gaussienne fausse.

**Limites** : le jeu date de 1999 et vient d'une seule boîte mail (variables `george`, `hp`, `650`) ; les
performances ne se transposeraient pas telles quelles à une autre messagerie. Le seuil de Neyman-Pearson est
estimé sur peu de faux positifs ; en production, il faudrait le recalibrer sur un plus grand échantillon de vrais e-mails.

Voir le notebook `diabetes.ipynb` pour le même workflow sur un jeu déséquilibré, et le `README.md` pour la
comparaison des deux jeux.